# Notebook 04 — Foundry Agent Pro-Code
## Contoso Banque — Churn Advisor Agent (Microsoft Foundry)

> **All data and scenarios are entirely synthetic and fictional.**
> "Contoso Banque" is a fictional entity used for educational purposes only.

---

## What This Notebook Does

This notebook creates the **Contoso Churn Advisor** agent using the **Azure AI Projects Python SDK**,
combining three tools: the Fabric Data Agent, Web Search, and Code Interpreter.

| Step | Description |
|---|---|
| Cell 1 | Install SDK (`azure-ai-projects` 2.x) |
| Cell 2 | Configure the Foundry project connection |
| Cell 3 | Define the tools — one cell each: Fabric, Web Search, Code Interpreter |
| Cell 4 | Assemble the tools and create the agent |
| — | Test the agent in the **Foundry portal** (no code) |
| Cell 5 | Cleanup — delete the agent (optional) |

## Prerequisites

- Microsoft Foundry project created (Step 10 of the workshop).
- A **full** model deployed (Step 11 — e.g. GPT-5.4; avoid "mini" for multi-tool orchestration).
- Fabric Data Agent configured with `ChurnAnalysisLH` tables.
- Foundry project **endpoint** (found in Project → Overview → Libraries → Foundry).

## Cell 1 — Install SDK and Configure Connection

In [2]:
# Install the required packages for the new Foundry Agents API.
# The new agent API (create_version / PromptAgentDefinition / MicrosoftFabricPreviewTool)
# requires azure-ai-projects 2.x. Run this cell once, then restart the kernel.
%pip install "azure-ai-projects>=2.0.0" azure-identity openai

Note: you may need to restart the kernel to use updated packages.


## Cell 2 — Configure Connection

Replace the `PROJECT_ENDPOINT` below with your Foundry project endpoint.

**Where to find it:**
1. Open [ai.azure.com](https://ai.azure.com)
2. Open your project (`contoso-churn-foundry`)
3. Go to **Overview** → **Libraries** → **Foundry**
4. Copy the **Endpoint** (format: `https://<name>.services.ai.azure.com/api/projects/<project>`)

In [3]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# === CONFIGURATION ===
# Replace with your Foundry project endpoint
# Format: https://<resource_name>.services.ai.azure.com/api/projects/<project_name>
PROJECT_ENDPOINT = "https://contoso-churn-foundry-resource.services.ai.azure.com/api/projects/contoso-churn-foundry"

# Model deployment name (from Step 11).
# NOTE: use a full model (e.g. gpt-5.4), not a "mini" — the agent model orchestrates the
# multi-tool routing, and mini models are unreliable at chaining Fabric + Web Search.
MODEL_DEPLOYMENT = "gpt-5.4"

# Name of your Fabric connection (Custom Keys connection created in Foundry:
# Management center → Connected resources → + New connection → Custom Keys)
FABRIC_CONNECTION_NAME = "ChurnFabricConnection"

# === CLIENTS ===
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
# In azure-ai-projects 2.x, get_openai_client() returns a plain OpenAI client
# (no api_version argument — that would be forwarded to OpenAI() and error out).
openai = project.get_openai_client()

print("✅ Connected to Microsoft Foundry project")

✅ Connected to Microsoft Foundry project


## Cell 3 — Define the Tools (one cell per tool)

The agent uses three tools. Each is built in its own cell below, then combined when the agent is created:

1. **Fabric Data Agent** — the single source of truth for Contoso Banque's internal churn data.
2. **Web Search** — external industry benchmarks, market trends, and retention strategies.
3. **Code Interpreter** — calculations, what-if simulations, and charts.

Run the three cells in order; each one prepares a tool object (`fabric_tool`, `web_search_tool`, `code_interpreter_tool`) that the agent-creation cell then assembles.

In [14]:
# Tool 1 — Fabric Data Agent (internal churn data = single source of truth)
from azure.ai.projects.models import (
    MicrosoftFabricPreviewTool,
    FabricDataAgentToolParameters,
    ToolProjectConnection,
)

# Resolve the Fabric connection ID from its name
fabric_connection = project.connections.get(FABRIC_CONNECTION_NAME)

fabric_tool = MicrosoftFabricPreviewTool(
    fabric_dataagent_preview=FabricDataAgentToolParameters(
        project_connections=[
            ToolProjectConnection(project_connection_id=fabric_connection.id)
        ]
    )
)

print("✅ Tool 1 ready: Fabric Data Agent")

✅ Tool 1 ready: Fabric Data Agent


In [15]:
# Tool 2 — Web Search (external benchmarks, market trends, retention strategies)
from azure.ai.projects.models import WebSearchTool, WebSearchApproximateLocation

web_search_tool = WebSearchTool(
    user_location=WebSearchApproximateLocation(
        country="FR", city="Paris", region="Île-de-France"
    )
)

print("✅ Tool 2 ready: Web Search")

✅ Tool 2 ready: Web Search


In [ ]:
# Tool 3 — Code Interpreter (calculations, what-if simulations, charts)
from azure.ai.projects.models import CodeInterpreterTool, AutoCodeInterpreterToolParam

# Uses an auto-provisioned sandbox container. No file upload is needed here — the tool
# computes on figures the agent already retrieved (e.g. from Fabric) and can draw charts.
code_interpreter_tool = CodeInterpreterTool(container=AutoCodeInterpreterToolParam())

print("✅ Tool 3 ready: Code Interpreter")

✅ Tool 3 ready: Code Interpreter


## Cell 4 — Create the Agent

Assemble the three tools (`fabric_tool`, `web_search_tool`, `code_interpreter_tool`) and create the agent version with its routing instructions.

In [17]:
# === AGENT INSTRUCTIONS ===
from azure.ai.projects.models import PromptAgentDefinition

INSTRUCTIONS = """You are the Churn Advisor for Contoso Banque, a fictional European retail bank.
Your role is to help the analytics team understand, analyse, and act on customer churn patterns.

TOOL ROUTING (read first — this is mandatory):
You have exactly three tools. Choose based on WHERE the answer lives:
- Fabric Data Agent tool → the SINGLE SOURCE OF TRUTH for anything about Contoso Banque's OWN
  customers, accounts, churn, segments, balances, products, counts, rates, or KPIs.
  You MUST call the Fabric Data Agent tool for ALL such internal questions.
  NEVER use Web Search to answer a question about Contoso Banque's own data.
  NEVER invent or estimate an internal figure — if the Fabric tool can't answer, say so.
- Web Search tool → ONLY for EXTERNAL information: industry benchmarks, market trends,
  competitor data, regulations, or retention strategies published on the web.
  Do NOT use Web Search for Contoso Banque's internal numbers.
- Code Interpreter tool → for CALCULATIONS on figures you already have: what-if simulations,
  projections, impact analysis, and charts. Feed it the numbers retrieved from Fabric.
  Do NOT use it to fetch data — only to compute or visualise.
- If a question needs several tools (e.g. "compare OUR churn to the industry and chart it"),
  it is a MULTI-STEP task. You MUST follow this order and MUST NOT skip step 1:
  STEP 1 (ALWAYS FIRST, MANDATORY): call the Fabric Data Agent tool to get the internal figure.
     Do this BEFORE any Web Search, even if the external part looks hard or you think you
     already know the answer. Any question containing "our", "we", "us", or "Contoso"
     REQUIRES a Fabric call as the first action.
  STEP 2: call Web Search for the external benchmark (if the question needs one).
  STEP 3: use Code Interpreter to compute or chart (if the question needs it).
  STEP 4: present the results, labelling each number [Internal] or [Web].
  If a later step fails, you MUST STILL report what you obtained in the earlier steps.
  NEVER abandon the whole answer just because one tool failed.

Examples:
- "What is our overall churn rate?" → Fabric only.
- "Which activity tier churns the most?" → Fabric only.
- "How many VIP customers do we have?" → Fabric only.
- "What is the average churn rate in European retail banking?" → Web Search only.
- "How does our churn compare to the industry?" → Fabric THEN Web Search.
- "If we cut Inactive churn from 42% to 25%, how many customers do we save?" → Fabric THEN Code Interpreter.
- "Chart our churn rate by activity tier." → Fabric THEN Code Interpreter.

KEY DEFINITIONS:
- 'churned_90d' = 1 means the customer has been flagged as churned in the last 90 days; = 0 means retained.
- 'activity_tier' groups customers by 90-day transaction frequency:
    "High" (>=10 txns), "Medium" (4-9 txns), "Low" (1-3 txns), "Inactive" (0 txns).
- 'balance_band' groups customers by avg 90-day balance:
    "High" (>=25000 EUR), "Medium" (5000-24999), "Low" (500-4999), "Very Low" (<500).
- 'product_count_tier' groups by number of active products:
    "Multi-product" (>=3), "Dual" (2), "Single" (1), "No product" (0).
- 'custom_segment' is a business-defined label from customer_custom_segment table:
    "VIP", "Loyal", "At Risk", "New Joiner", "Dormant".
  To query custom segments, JOIN customer_custom_segment ON customer_id with customer_360.

DATA CONTEXT:
- All monetary amounts are in EUR.
- All data is synthetic and fictional.

BEHAVIOUR:
- Clearly label which insights come from internal data ([Internal]) vs. external web sources ([Web]).
- Always show churn rates as percentages.
- Round monetary amounts to the nearest euro.
- Be concise and actionable.
"""

# === CREATE THE AGENT (Fabric + Web Search + Code Interpreter) ===
# NOTE: agent_name must be alphanumeric with hyphens only (no spaces), max 63 chars.
agent = project.agents.create_version(
    agent_name="ContosoChurnAdvisor",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=INSTRUCTIONS,
        tools=[fabric_tool, web_search_tool, code_interpreter_tool],
    ),
    description="Contoso Banque churn advisor with Fabric data + web search + code interpreter.",
)

print(f"✅ Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

✅ Agent created (id: ContosoChurnAdvisor:1, name: ContosoChurnAdvisor, version: 1)


## Test the Agent in the Foundry Portal

You don't need to test the agent from this notebook — do it interactively in the **Foundry portal**, which also runs the Fabric tool under **your** signed-in identity (identity passthrough).

1. Open [ai.azure.com](https://ai.azure.com) → your project (`contoso-churn-foundry`).
2. Go to **Agents** and select **`ContosoChurnAdvisor`** (the agent created above). Pick the **latest version**.
3. Open the **Playground** (Try / Test) pane.
4. Ask sample questions, one per tool and one multi-tool:
   - *"What is our overall churn rate?"* → **Fabric** only.
   - *"What is the average churn rate in European retail banking?"* → **Web Search** only.
   - *"If we cut Inactive churn from 42% to 25%, how many customers do we save?"* → **Fabric → Code Interpreter**.
   - *"How does our churn compare to the industry, and chart it?"* → **Fabric → Web Search → Code Interpreter**.
5. Open the **Traces** tab to confirm the expected tools were called in order.

> 💡 Use a **full** model (e.g. GPT-5.4) for the agent — "mini" models are unreliable at chaining multiple tools. If a data question skips Fabric, switch the model in the playground **Model** dropdown and Save.

## Cell 5 — Cleanup (Optional)

Delete the agent when you're done to avoid unnecessary resource usage.
**Only run this cell if you want to remove the agent.**

In [18]:
# ⚠️ CAUTION: This deletes the agent version permanently.
# Uncomment the lines below to execute.

# project.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
# print("✅ Agent deleted")